In [7]:
pip install opendatasets

Note: you may need to restart the kernel to use updated packages.


d:\CA_content\Python\Final_Project\.venv\Scripts\python.exe: No module named pip


### IMPORTING_THE_DATA

In [8]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/data")

ModuleNotFoundError: No module named 'cgi'

### Concatenation of dataframe

In [ ]:
import pandas as pd
import gc 
from IPython.display import display

print("⏳ Loading Datasets (Raw Mode - No Data Type Conversion, No Product_ID)...")

# 1. File paths
path_oct = "ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv"
path_nov = "ecommerce-behavior-data-from-multi-category-store/2019-Nov.csv"

# 2. Updated Columns (product_id aur category_id hata diye)
columns_to_keep = ['event_time', 'event_type', 'category_code', 'brand', 'price', 'user_id', 'user_session']

# 3. Data Read Karna (Bina kisi datatype conversion ke)
print("Reading October Data...")
df_oct = pd.read_csv(path_oct, usecols=columns_to_keep)
print(f"✅ October Data Loaded: {df_oct.shape}")

print("Reading November Data...")
df_nov = pd.read_csv(path_nov, usecols=columns_to_keep)
print(f"✅ November Data Loaded: {df_nov.shape}")

# 4. Dono DataFrames ko Concat (Jodna)
print("🔗 Concatenating DataFrames...")
final_df = pd.concat([df_oct, df_nov], ignore_index=True)

# 5. RAM ko turant free karna (Garbage Collection)
del df_oct
del df_nov
gc.collect()

print(f"🎉 Final DataFrame Ready! Shape: {final_df.shape}")
display(final_df.head())

#Conversion of Data-time

In [ ]:
final_df['event_time'] = pd.to_datetime(final_df['event_time'])

In [ ]:
final_df.info()

#### Data-Preprocessing Pipeline

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

print("⚙️ Initiating Ultimate Master Data Pipeline (With Anti-Bot, Capping & Super Features)...")

# ==========================================
# STEP 1: PREPARATION & MEMORY OPTIMIZATION
# ==========================================
# Encoding event types for faster processing
final_df['is_view'] = (final_df['event_type'] == 'view').astype(np.int8)
final_df['is_cart'] = (final_df['event_type'] == 'cart').astype(np.int8)
final_df['is_purchase'] = (final_df['event_type'] == 'purchase').astype(np.int8)

# Prices split (Prevents Data Leakage & helps in Budget Clarity)
final_df['view_price'] = np.where(final_df['event_type'] == 'view', final_df['price'], np.nan)
final_df['cart_price'] = np.where(final_df['event_type'] == 'cart', final_df['price'], np.nan)

# Time to First Cart Prep
first_cart_time = final_df[final_df['is_cart'] == 1].groupby('user_session')['event_time'].min()

# ==========================================
# STEP 2: THE MASTER GROUPBY
# ==========================================
ml_dataframe = final_df.groupby('user_session').agg(
    total_views=('is_view', 'sum'),
    total_carts=('is_cart', 'sum'),
    unique_products=('product_id', 'nunique'),
    
    median_view_price=('view_price', 'median'),  
    median_cart_price=('cart_price', 'median'), 
    
    # Min and Max view prices for Budget Clarity
    max_view_price=('view_price', 'max'),
    min_view_price=('view_price', 'min'),
    
    start_time=('event_time', 'min'),
    end_time=('event_time', 'max'),
    target_purchase=('is_purchase', 'max')
)

# Fill missing values
ml_dataframe[['median_view_price', 'median_cart_price']] = ml_dataframe[['median_view_price', 'median_cart_price']].fillna(0)
ml_dataframe[['max_view_price', 'min_view_price']] = ml_dataframe[['max_view_price', 'min_view_price']].fillna(0)

# Raw Duration Calculation
ml_dataframe['session_duration_seconds'] = (ml_dataframe['end_time'] - ml_dataframe['start_time']).dt.total_seconds()

# ==========================================
# STEP 3: DROP THE BOTS (Using Raw Data)
# ==========================================
print(f"Original Data Size: {ml_dataframe.shape[0]:,}")

# Temporary raw speed for checking bots
raw_duration_minutes = (ml_dataframe['session_duration_seconds'] + 1) / 60
raw_views_per_minute = ml_dataframe['total_views'] / raw_duration_minutes

# Drop condition (Fast-Clicking Bots)
bot_condition = (raw_views_per_minute > 30) & (ml_dataframe['total_views'] > 15)
ml_dataframe = ml_dataframe[~bot_condition] 

print(f"Data Size after removing Bots: {ml_dataframe.shape[0]:,}")

# ==========================================
# STEP 4: CAP THE ZOMBIE HUMANS
# ==========================================
ml_dataframe['session_duration_seconds'] = np.clip(
    ml_dataframe['session_duration_seconds'], 
    a_min=0, 
    a_max=3600 # 1 Hour cap
)
print(f"Maximum session duration capped to: {ml_dataframe['session_duration_seconds'].max()} seconds")

# ==========================================
# STEP 5: FINAL SUPER FEATURES 
# ==========================================
# 1. Base Features
ml_dataframe['views_per_minute'] = ml_dataframe['total_views'] / ((ml_dataframe['session_duration_seconds'] + 1) / 60)
ml_dataframe['focus_ratio'] = ml_dataframe['total_views'] / ml_dataframe['unique_products'].replace(0, 1)

# 2. FEATURE I: Time to First Cart
ml_dataframe['first_cart_time_seconds'] = (first_cart_time - ml_dataframe['start_time']).dt.total_seconds()
ml_dataframe['first_cart_time_seconds'] = ml_dataframe['first_cart_time_seconds'].fillna(0)
ml_dataframe['first_cart_time_seconds'] = np.clip(ml_dataframe['first_cart_time_seconds'], a_min=0, a_max=3600)

# 3. FEATURE II: Post-Cart Hesitation (Dwell Time)
ml_dataframe['post_cart_dwell_seconds'] = np.where(
    ml_dataframe['total_carts'] > 0,
    ml_dataframe['session_duration_seconds'] - ml_dataframe['first_cart_time_seconds'],
    0
)
ml_dataframe['post_cart_dwell_seconds'] = np.clip(ml_dataframe['post_cart_dwell_seconds'], a_min=0, a_max=3600)

# 4. FEATURE III: Budget Clarity (Price Exploration Range)
ml_dataframe['price_exploration_range'] = ml_dataframe['max_view_price'] - ml_dataframe['min_view_price']
CAP_PRICE = 1000
ml_dataframe['price_exploration_capped'] = np.clip(ml_dataframe['price_exploration_range'], a_min=0, a_max=CAP_PRICE)

# 5. FEATURE IV: Decisiveness Metric (Cart to View Ratio)
# Formula: Total_Carts / Total_Views
# Replacing 0 views with 1 to avoid ZeroDivisionError
ml_dataframe['cart_view_ratio'] = ml_dataframe['total_carts'] / ml_dataframe['total_views'].replace(0, 1)
# Capping at 1.0 just in case there are rare sessions with more carts than views
ml_dataframe['cart_view_ratio'] = np.clip(ml_dataframe['cart_view_ratio'], a_min=0, a_max=1.0)

# ==========================================
# STEP 6: CLEANUP
# ==========================================
# Drop intermediate columns
columns_to_drop = ['start_time', 'end_time', 'max_view_price', 'min_view_price', 'price_exploration_range']
ml_dataframe = ml_dataframe.drop(columns=columns_to_drop)

print("\n✅ Final ml_dataframe is perfectly engineered, cleaned, and ready for ML!")
display(ml_dataframe.head())

### Wrapper Method for Best-features for XGBoost

In [ ]:
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc, f1_score, recall_score
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS

sns.set_theme(style="whitegrid")

# Puraane 'Unnamed' columns agar hain toh unhe safely hatana
X_full = ml_dataframe.loc[:, ~ml_dataframe.columns.str.contains('^Unnamed')].drop(columns=['target_purchase'])
y_full = ml_dataframe['target_purchase']

# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42, stratify=y_full)
imbalance_weight = (y_train == 0).sum() / (y_train == 1).sum()

print("=" * 70)
print("🔍 PHASE 1: EXHAUSTIVE FEATURE SELECTION (Optimizing for RECALL)")
print("=" * 70)

# Time bachane ke liye Training data ka 10% sample le rahe hain sirf EFS ke liye
X_train_sample, _, y_train_sample, _ = train_test_split(X_train, y_train, train_size=0.4, random_state=42, stratify=y_train)

# Base model for selection
xgb_base = XGBClassifier(scale_pos_weight=imbalance_weight, random_state=42, n_jobs=-1)

# EFS Setup (Min 2 columns se lekar saare columns tak check karega)
efs = EFS(xgb_base, 
          min_features=2,
          max_features=X_train.shape[1],
          scoring='recall', # Specially prioritizing Recall
          print_progress=True,
          cv=2, # Fast evaluation
          n_jobs=-1)

print(f"Running EFS on {len(X_train_sample)} rows to find the best feature combination...")
efs = efs.fit(X_train_sample, y_train_sample)

best_features = list(efs.best_feature_names_)
print(f"\n🏆 BEST FEATURES FOR RECALL: {best_features}")
print(f"🔥 EFS Best Recall Score: {efs.best_score_:.4f}\n")

# Ab hum apne X_train aur X_test ko sirf inhi best columns tak limit kar denge
X_train_best = X_train[best_features]
X_test_best = X_test[best_features]


### Training of XGBoost

In [ ]:
print("=" * 70)
print("⚙️ PHASE 2: ADVANCED HYPERPARAMETER TUNING (Deep Grid)")
print("=" * 70)

# 🚀 THE ULTIMATE HYPERPARAMETER GRID
param_grid = {
    'n_estimators': [100, 200, 300, 400],              # Kitne trees banayega (Zyada trees + kam learning rate = Best)
    'max_depth': [3, 5, 7, 9],                         # Tree kitna gehra hoga
    'learning_rate': [0.01, 0.05, 0.1, 0.2],           # Seekhne ki speed
    'subsample': [0.7, 0.8, 0.9, 1.0],                 # Overfitting rokne ke liye rows ka fraction
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],          # Overfitting rokne ke liye columns ka fraction
    'gamma': [0, 0.5, 1, 2],                           # Strictness for splitting
    'scale_pos_weight': [imbalance_weight, imbalance_weight * 1.2, imbalance_weight * 1.5] # Pushing hard for Recall
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# RandomizedSearch Setup (Optimizing strictly for Recall)
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=30,           # MAGIC NUMBER: Pehle 5 tha, ab hum 30 random combinations try karenge!
    scoring='recall',    
    cv=skf,
    verbose=2,           # Isey 2 kar diya hai taaki aapko screen par live progress dikhe
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
print(f"Starting Cross-Validation Tuning (Trying {random_search.n_iter} combinations, totalling {random_search.n_iter * 3} fits)...")
random_search.fit(X_train_best, y_train)
print(f"✅ Tuning Completed in {(time.time() - start_time)/60:.2f} minutes!\n")

champion_model = random_search.best_estimator_
print(f"🏆 WINNING HYPERPARAMETERS: {random_search.best_params_}")


print("=" * 70)
print("📈 PHASE 3: COMPREHENSIVE EVALUATION (ROC-AUC & MATRIX)")
print("=" * 70)

# Predictions nikalna
y_pred = champion_model.predict(X_test_best)
y_probs = champion_model.predict_proba(X_test_best)[:, 1]

# 1. SCORES PRINT KARNA
print("1. OVERALL SCORES:")
print(f"-> Accuracy Score: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"-> Recall Score (Class 1): {recall_score(y_test, y_pred)*100:.2f}%")
print(f"-> F1 Score (Class 1): {f1_score(y_test, y_pred)*100:.2f}%\n")

# 2. CLASSIFICATION REPORT
print("2. DETAILED CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))

# 3. GRAPHS (Confusion Matrix & ROC-AUC)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Graph A: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', 
            xticklabels=['Not Purchased (0)', 'Purchased (1)'], 
            yticklabels=['Not Purchased (0)', 'Purchased (1)'], ax=ax1)
ax1.set_title('Final Confusion Matrix', fontsize=15, fontweight='bold')
ax1.set_xlabel('Predicted by Champion Model', fontsize=12)
ax1.set_ylabel('Actual Truth', fontsize=12)

# Graph B: ROC-AUC Curve
fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

ax2.plot(fpr, tpr, color='#DD8452', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random Guess') 
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax2.set_title('ROC-AUC Curve', fontsize=15, fontweight='bold')
ax2.legend(loc="lower right", fontsize=12)

plt.tight_layout()
plt.show()

### Data-Preprocessing for GPUs

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Masking
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

print("⚙️ Initiating GRU Data Transformation Pipeline (Colab GPU Optimized)...")

# --- DATA PREPARATION PHASE ---

# 1. Sort by session and time
seq_df = final_df.sort_values(by=['user_session', 'event_time']).copy()

# 2. Target Creation (1 if purchased, else 0)
session_targets = seq_df.groupby('user_session')['event_type'].apply(
    lambda events: 1 if 'purchase' in events.values else 0
).to_dict()

# 3. Remove 'purchase' events from input sequence to prevent cheating
seq_df = seq_df[seq_df['event_type'] != 'purchase'].copy()

# 4. Feature Engineering
seq_df['is_view'] = (seq_df['event_type'] == 'view').astype(int)
seq_df['is_cart'] = (seq_df['event_type'] == 'cart').astype(int)
seq_df['time_gap_seconds'] = seq_df.groupby('user_session')['event_time'].diff().dt.total_seconds().fillna(0)
seq_df['log_time_gap'] = np.log1p(seq_df['time_gap_seconds'])

scaler = MinMaxScaler()
seq_df['scaled_price'] = scaler.fit_transform(seq_df[['price']])

sequence_features = ['is_view', 'is_cart', 'scaled_price', 'log_time_gap']

print("Grouping events into sequential lists...")
session_groups = seq_df.groupby('user_session')[sequence_features].apply(lambda x: x.values.tolist())
y = np.array([session_targets[session_id] for session_id in session_groups.index])

# 5. SEQUENCE PADDING (THE FIX FOR CUDNN)
MAX_STEPS = 15
print(f"Padding sequences to {MAX_STEPS} steps (Using 'post' padding for cuDNN)...")
# Changed padding from 'pre' to 'post'
X = pad_sequences(session_groups.tolist(), maxlen=MAX_STEPS, padding='post', truncating='pre', dtype='float32')

# 6. DATA SANITIZATION (Safeguard against NaNs)
print("🧹 Sanitizing Data...")
X = np.asarray(X).astype(np.float32)
y = np.asarray(y).astype(np.float32)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print(f"🔹 Final Shape of Input (X): {X.shape}")
print(f"🔹 Final Shape of Target (y): {y.shape}")
print("-" * 50)


# --- MODEL BUILDING & TRAINING PHASE ---

print("🧠 Building the GRU Neural Network...")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = Sequential()

# Layer 0: Explicit Input Layer
model.add(tf.keras.Input(shape=(X.shape[1], X.shape[2])))

# Layer 1: Masking Layer
model.add(Masking(mask_value=0.0))

# Layer 2: GRU Layer (Will now safely use cuDNN)
model.add(GRU(units=64, return_sequences=False, activation='tanh'))

# Layer 3: Dropout
model.add(Dropout(0.2))

# Layer 4: Dense
model.add(Dense(units=32, activation='relu'))

# Layer 5: Output
model.add(Dense(units=1, activation='sigmoid'))

print("⚙️ Compiling the model...")
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

early_stop = EarlyStopping(monitor='val_auc', mode='max', patience=3, restore_best_weights=True)

print("\n🚀 Starting Training...")
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop]
)

print("\n📊 Evaluating on Test Data...")
test_loss, test_acc, test_auc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC Score: {test_auc:.4f}")

### Test_Cases

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("🧪 RUNNING GRU PSYCHOLOGY TEST CASES...\n")

# Helper function time gap ko log mein badalne ke liye
def time_to_log(seconds):
    return np.log1p(seconds)

# Features ka order: [is_view, is_cart, scaled_price, log_time_gap]
# Note: Price ko hum seedha 0.0 (Cheap) se 1.0 (Expensive) ke beech likh rahe hain

# -----------------------------------------------------------------------------
# TEST CASE 1: "The VIP Focused Buyer"
# Behavior: Mehenga item dekha, turant cart mein dala (Low hesitation).
# Expectation: Very High Probability (> 0.80)
# -----------------------------------------------------------------------------
buyer_vip = [
    [1, 0, 0.9, time_to_log(0)],   # Step 1: View phone
    [1, 0, 0.9, time_to_log(15)],  # Step 2: View same phone again after 15 secs
    [0, 1, 0.9, time_to_log(10)]   # Step 3: Add to cart after 10 secs
]

# -----------------------------------------------------------------------------
# TEST CASE 2: "The Window Shopper (Timepass)"
# Behavior: Sasti cheezein dekh raha hai, bohot lamba time le raha hai, no cart.
# Expectation: Very Low Probability (< 0.20)
# -----------------------------------------------------------------------------
buyer_window = [
    [1, 0, 0.1, time_to_log(0)],    # Step 1: View cheap case
    [1, 0, 0.2, time_to_log(120)],  # Step 2: View another case (2 mins later)
    [1, 0, 0.1, time_to_log(300)],  # Step 3: View again (5 mins later)
    [1, 0, 0.3, time_to_log(400)]   # Step 4: View again (6.5 mins later)
]

# -----------------------------------------------------------------------------
# TEST CASE 3: "The Cart Abandoner (Drop-off Risk)"
# Behavior: Cart mein dala, par uske baad achanak bohot bada time gap aa gaya (Fas gaya).
# Expectation: Moderate to Low Probability (GRU should catch the hesitation!)
# -----------------------------------------------------------------------------
buyer_abandoner = [
    [1, 0, 0.5, time_to_log(0)],    # Step 1: View Mid-range item
    [0, 1, 0.5, time_to_log(30)],   # Step 2: Add to cart (Good sign!)
    [1, 0, 0.5, time_to_log(1800)]  # Step 3: View homepage again after 30 MINUTES (Bad sign!)
]

# --- PREPARING DATA FOR MODEL ---
MAX_STEPS = 15

# Pad the sequences (Must use 'post' padding to match training data)
test_sequences = [buyer_vip, buyer_window, buyer_abandoner]
X_test_cases = pad_sequences(test_sequences, maxlen=MAX_STEPS, padding='post', dtype='float32')

# Get Predictions
predictions = model.predict(X_test_cases, verbose=0)

# --- PRINTING RESULTS ---
print("📊 TEST RESULTS:")
print("-" * 50)
print(f"1. VIP Focused Buyer  : {predictions[0][0]*100:.2f}% Chance of Purchase")
print(f"2. Window Shopper     : {predictions[1][0]*100:.2f}% Chance of Purchase")
print(f"3. Cart Abandoner     : {predictions[2][0]*100:.2f}% Chance of Purchase")
print("-" * 50)